# 📱 Taller 02 — Clasificación: Telco Customer Churn
**Maestría en Ciencia de Datos | EAFIT — Periodo 2026-1**  
Curso: Inteligencia Artificial ECA&I | Docente: Jorge Iván Padilla-Buriticá

---
### Modelos evaluados
| # | Modelo |
|---|---|
| 1 | Regresión Logística |
| 2 | Random Forest Classifier |
| 3 | Gradient Boosting Classifier |
| 4 | K-Nearest Neighbors Classifier |

### Métricas de evaluación
| Métrica | Descripción |
|---|---|
| **F1-Score**  | Media armónica de Precision y Recall — clave en clases desbalanceadas |
| **Accuracy**  | % de predicciones correctas (interpretar con cuidado por desbalance) |
| **Precision** | De los predichos como Churn, ¿cuántos realmente lo son? |
| **Recall**    | De los Churn reales, ¿cuántos detectamos? (sensibilidad) |
| **ROC-AUC**   | Capacidad discriminativa en todos los umbrales posibles |

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, StratifiedKFold)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    classification_report
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
print('✅ Librerías cargadas correctamente')

## 2. Carga de Datos

**Dataset:** [Telco Customer Churn — Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

| Variable | Tipo | Descripción |
|---|---|---|
| `tenure` | Numérica | Meses como cliente |
| `MonthlyCharges` | Numérica | Cargo mensual (USD) |
| `TotalCharges` | Numérica | Cargo total acumulado |
| `Contract` | Categórica | Tipo de contrato |
| `InternetService` | Categórica | Tipo de internet |
| `PaymentMethod` | Categórica | Método de pago |
| `Churn` | Binaria | **TARGET: 1=Abandona, 0=Se queda** |

In [ ]:
# ── Opción 1: Dataset real de Kaggle ─────────────────────────────────
# df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
# df['Churn'] = (df['Churn'] == 'Yes').astype(int)
# df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
# df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# ── Opción 2: Datos sintéticos (ejecutar si no se tiene el CSV) ────────
np.random.seed(RANDOM_STATE)
n = 7043
tenure   = np.random.randint(0, 72, n)
monthly  = np.round(np.random.uniform(20, 110, n), 2)
total    = np.round(tenure * monthly + np.random.normal(0,50,n), 2).clip(0)
contract = np.random.choice(['Month-to-month','One year','Two year'], n, p=[0.55,0.24,0.21])
internet = np.random.choice(['DSL','Fiber optic','No'], n, p=[0.34,0.44,0.22])
payment  = np.random.choice(['Electronic check','Mailed check',
                               'Bank transfer','Credit card'], n)
senior   = np.random.choice([0,1], n, p=[0.84,0.16])
partner  = np.random.choice(['Yes','No'], n)
paperless= np.random.choice(['Yes','No'], n, p=[0.59,0.41])
churn_p  = (0.05 + 0.30*(contract=='Month-to-month')
             + 0.10*(internet=='Fiber optic')
             + 0.08*(payment=='Electronic check')
             - 0.008*tenure
             + np.random.normal(0,0.05,n)).clip(0,1)
churn = (np.random.uniform(0,1,n) < churn_p).astype(int)

df = pd.DataFrame({
    'tenure':tenure,'MonthlyCharges':monthly,'TotalCharges':total,
    'Contract':contract,'InternetService':internet,'PaymentMethod':payment,
    'SeniorCitizen':senior,'Partner':partner,'PaperlessBilling':paperless,
    'Churn':churn
})

print(f'Shape: {df.shape}')
print(f'Churn rate: {df["Churn"].mean():.1%} — Clases desbalanceadas')
df.head(10)

## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Estadísticas Descriptivas y Valores Faltantes

In [ ]:
print('=== INFORMACIÓN GENERAL ===')
df.info()
print('\n=== ESTADÍSTICAS DESCRIPTIVAS ===')
display(df.describe().round(2))
print('\n=== VALORES FALTANTES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '✅ Sin valores faltantes')
print('\n=== DISTRIBUCIÓN DEL TARGET ===')
vc = df['Churn'].value_counts()
print(f'  No Churn (0): {vc[0]} ({vc[0]/len(df):.1%})')
print(f'  Churn    (1): {vc[1]} ({vc[1]/len(df):.1%})')
print('\n⚠️ Dataset desbalanceado → usar F1, ROC-AUC como métricas principales')

### 3.2 Visualización del Desbalance y Variables Clave

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Distribución Churn
vc = df['Churn'].value_counts()
axes[0,0].bar(['No Churn (0)','Churn (1)'], vc.values,
               color=['steelblue','tomato'], edgecolor='white', width=0.5)
axes[0,0].set_title('Distribución del Target (Churn)', fontsize=12, fontweight='bold')
for i, v in enumerate(vc.values):
    axes[0,0].text(i, v+60, f'{v}\n({v/len(df):.1%})', ha='center', fontsize=10)

# Churn rate por Contract
ct = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
axes[0,1].bar(ct.index, ct.values, color=['tomato','steelblue','steelblue'], edgecolor='white')
axes[0,1].set_title('Tasa de Churn por Contrato', fontsize=12, fontweight='bold')
axes[0,1].set_ylabel('Tasa de Churn')
axes[0,1].tick_params(axis='x', rotation=20)
for i, v in enumerate(ct.values):
    axes[0,1].text(i, v+0.005, f'{v:.1%}', ha='center')

# Churn por InternetService
ci = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False)
axes[0,2].bar(ci.index, ci.values, color='mediumpurple', edgecolor='white')
axes[0,2].set_title('Tasa de Churn por Internet', fontsize=12, fontweight='bold')
axes[0,2].set_ylabel('Tasa de Churn')
for i, v in enumerate(ci.values):
    axes[0,2].text(i, v+0.005, f'{v:.1%}', ha='center')

# Tenure por Churn
for label, grp in df.groupby('Churn'):
    axes[1,0].hist(grp['tenure'], bins=30, alpha=0.6,
                    label='Churn' if label==1 else 'No Churn',
                    color='tomato' if label==1 else 'steelblue')
axes[1,0].set_title('Tenure por Churn', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('Meses'); axes[1,0].legend()

# MonthlyCharges por Churn
for label, grp in df.groupby('Churn'):
    axes[1,1].hist(grp['MonthlyCharges'], bins=30, alpha=0.6,
                    label='Churn' if label==1 else 'No Churn',
                    color='tomato' if label==1 else 'steelblue')
axes[1,1].set_title('MonthlyCharges por Churn', fontsize=12, fontweight='bold')
axes[1,1].legend()

# Heatmap
corr = df[['tenure','MonthlyCharges','TotalCharges','SeniorCitizen','Churn']].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            ax=axes[1,2], square=True, linewidths=0.5)
axes[1,2].set_title('Correlaciones', fontsize=12, fontweight='bold')

plt.suptitle('EDA — Telco Customer Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_churn.png', bbox_inches='tight')
plt.show()

## 4. Preprocesamiento y Feature Engineering

### Estrategia para clases desbalanceadas
- `class_weight='balanced'` en los clasificadores que lo soportan
- `StratifiedKFold` para preservar la proporción de clases en cada fold
- Métricas principales: F1-Score y ROC-AUC (no Accuracy)

### Features engineered
| Feature | Descripción |
|---|---|
| `charges_per_month` | TotalCharges / (tenure + 1) — eficiencia de gasto |
| `is_month_to_month` | Flag de contrato mes a mes (mayor riesgo de churn) |
| `high_monthly` | Flag de cargo mensual alto (> percentil 75) |

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────
df_fe = df.copy()
df_fe['charges_per_month'] = df_fe['TotalCharges'] / (df_fe['tenure'] + 1)
df_fe['is_month_to_month'] = (df_fe['Contract'] == 'Month-to-month').astype(int)
df_fe['high_monthly']      = (df_fe['MonthlyCharges'] > df_fe['MonthlyCharges'].quantile(0.75)).astype(int)

print('Features creadas: charges_per_month, is_month_to_month, high_monthly')

# ── Definición de features ───────────────────────────────────────────
FEATURES  = ['tenure','MonthlyCharges','TotalCharges','SeniorCitizen',
              'Contract','InternetService','PaymentMethod','Partner',
              'PaperlessBilling','charges_per_month','is_month_to_month','high_monthly']
TARGET    = 'Churn'
num_cols  = ['tenure','MonthlyCharges','TotalCharges','charges_per_month']
cat_cols  = ['Contract','InternetService','PaymentMethod','Partner','PaperlessBilling']
pass_cols = ['SeniorCitizen','is_month_to_month','high_monthly']

X = df_fe[FEATURES]
y = df_fe[TARGET]

# ── Split estratificado 80/20 ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Churn en train: {y_train.mean():.1%} | en test: {y_test.mean():.1%}')

# ── Preprocesador ────────────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num',  StandardScaler(), num_cols),
    ('cat',  OneHotEncoder(drop='first', sparse_output=False,
                            handle_unknown='ignore'), cat_cols),
    ('pass', 'passthrough', pass_cols)
])
print('✅ Preprocesador definido (fit solo sobre X_train)')

## 5. Definición y Entrenamiento de Modelos

| Modelo | Justificación |
|---|---|
| **Regresión Logística** | Baseline interpretable, calibrado probabilísticamente, soporta `class_weight='balanced'` |
| **Random Forest** | Ensemble de árboles con bagging, maneja desbalance, da Feature Importance |
| **Gradient Boosting** | Entrena secuencialmente corrigiendo errores; muy efectivo en tabular data |
| **KNN Classifier** | No paramétrico, útil como comparación, requiere escalado |

In [ ]:
# ── Pipelines ─────────────────────────────────────────────────────────
pipelines = {
    'Regresión Logística': Pipeline([
        ('prep',  preprocessor),
        ('model', LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('prep',  preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'Gradient Boosting': Pipeline([
        ('prep',  preprocessor),
        ('model', GradientBoostingClassifier(
            n_estimators=100, random_state=RANDOM_STATE))
    ]),
    'KNN Classifier': Pipeline([
        ('prep',  preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=15))
    ]),
}
print('✅ Pipelines definidos para 4 modelos')

## 6. Validación Cruzada (StratifiedKFold — 5 folds)

Se usa `StratifiedKFold` para mantener la proporción de clases (≈26% churn)
en cada fold de entrenamiento y validación.
Métricas calculadas en CV: F1, ROC-AUC, Accuracy, Precision, Recall

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

print(f'{'Modelo':<22} {'CV F1':>8} {'CV AUC':>9} {'CV Acc':>8} {'CV Prec':>9} {'CV Rec':>8}')
print('-' * 70)

for name, pipe in pipelines.items():
    f1   = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='f1')
    auc  = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='roc_auc')
    acc  = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='accuracy')
    prec = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='precision')
    rec  = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='recall')
    cv_results[name] = {
        'CV F1_mean':        round(f1.mean(),   4),
        'CV F1_std':         round(f1.std(),    4),
        'CV AUC_mean':       round(auc.mean(),  4),
        'CV Accuracy_mean':  round(acc.mean(),  4),
        'CV Precision_mean': round(prec.mean(), 4),
        'CV Recall_mean':    round(rec.mean(),  4),
    }
    print(f'{name:<22} {f1.mean():>8.4f} {auc.mean():>9.4f} {acc.mean():>8.4f} '
          f'{prec.mean():>9.4f} {rec.mean():>8.4f}')

## 7. Ajuste de Hiperparámetros — GridSearchCV

Se optimizan los 4 modelos usando `scoring='roc_auc'` como criterio principal.

In [ ]:
# ── GridSearch Regresión Logística ───────────────────────────────────
param_lr = {
    'model__C':       [0.01, 0.1, 1.0, 10.0],
    'model__penalty': ['l1', 'l2'],
    'model__solver':  ['liblinear'],
}
grid_lr = GridSearchCV(
    Pipeline([('prep', preprocessor),
              ('model', LogisticRegression(max_iter=1000, class_weight='balanced',
                                            random_state=RANDOM_STATE))]),
    param_lr, cv=skf, scoring='roc_auc', n_jobs=-1, verbose=0)
grid_lr.fit(X_train, y_train)
print(f'LR  — Mejores parámetros: {grid_lr.best_params_}')
print(f'      Mejor CV AUC:       {grid_lr.best_score_:.4f}')

In [ ]:
# ── GridSearch Random Forest ─────────────────────────────────────────
param_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth':    [None, 10, 20],
    'model__max_features': ['sqrt', 'log2'],
}
grid_rf = GridSearchCV(
    Pipeline([('prep', preprocessor),
              ('model', RandomForestClassifier(class_weight='balanced',
                                               random_state=RANDOM_STATE))]),
    param_rf, cv=skf, scoring='roc_auc', n_jobs=-1, verbose=0)
grid_rf.fit(X_train, y_train)
print(f'RF  — Mejores parámetros: {grid_rf.best_params_}')
print(f'      Mejor CV AUC:       {grid_rf.best_score_:.4f}')

In [ ]:
# ── GridSearch Gradient Boosting ─────────────────────────────────────
param_gb = {
    'model__n_estimators':  [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth':     [3, 5],
}
grid_gb = GridSearchCV(
    Pipeline([('prep', preprocessor),
              ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))]),
    param_gb, cv=skf, scoring='roc_auc', n_jobs=-1, verbose=0)
grid_gb.fit(X_train, y_train)
print(f'GB  — Mejores parámetros: {grid_gb.best_params_}')
print(f'      Mejor CV AUC:       {grid_gb.best_score_:.4f}')

In [ ]:
# ── GridSearch KNN ───────────────────────────────────────────────────
param_knn = {
    'model__n_neighbors': [10, 15, 20, 30],
    'model__weights':     ['uniform', 'distance'],
    'model__p':           [1, 2],
}
grid_knn = GridSearchCV(
    Pipeline([('prep', preprocessor),
              ('model', KNeighborsClassifier())]),
    param_knn, cv=skf, scoring='roc_auc', n_jobs=-1, verbose=0)
grid_knn.fit(X_train, y_train)
print(f'KNN — Mejores parámetros: {grid_knn.best_params_}')
print(f'      Mejor CV AUC:       {grid_knn.best_score_:.4f}')

## 8. Evaluación Final en Test Set

Métricas calculadas sobre datos nunca vistos:
**F1-Score · Accuracy · Precision · Recall · ROC-AUC**

In [ ]:
best_models = {
    'Regresión Logística': grid_lr.best_estimator_,
    'Random Forest':       grid_rf.best_estimator_,
    'Gradient Boosting':   grid_gb.best_estimator_,
    'KNN Classifier':      grid_knn.best_estimator_,
}

test_results = {}
predictions  = {}
probabilities = {}

print(f'{'Modelo':<22} {'F1':>7} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'AUC':>8}')
print('=' * 70)

for name, mdl in best_models.items():
    y_pred = mdl.predict(X_test)
    y_prob = mdl.predict_proba(X_test)[:,1]
    predictions[name]   = y_pred
    probabilities[name] = y_prob
    f1   = f1_score(y_test, y_pred)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    test_results[name] = {
        'F1-Score':  round(f1,   4),
        'Accuracy':  round(acc,  4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
        'ROC-AUC':   round(auc,  4),
    }
    print(f'{name:<22} {f1:>7.4f} {acc:>9.4f} {prec:>10.4f} {rec:>8.4f} {auc:>8.4f}')

In [ ]:
# ── Tabla resumen con CV + Test ──────────────────────────────────────
rows = []
for name in best_models:
    row = {'Modelo': name}
    row['CV F1']       = cv_results[name]['CV F1_mean']
    row['CV AUC']      = cv_results[name]['CV AUC_mean']
    row.update({f'Test {k}': v for k, v in test_results[name].items()})
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Modelo')
highlight_cols = list(summary_df.columns)
display(summary_df.style
    .highlight_max(subset=highlight_cols, color='#c8e6c9')
    .format('{:.4f}')
)

## 9. Visualizaciones de Resultados

### 9.1 Comparación de las 5 Métricas entre Modelos

In [ ]:
metrics_order = ['F1-Score','Accuracy','Precision','Recall','ROC-AUC']
model_names   = list(test_results.keys())
colors = ['#1a73e8','#34a853','#ea4335','#ff9800']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()

for i, metric in enumerate(metrics_order):
    ax   = axes_flat[i]
    vals = [test_results[n][metric] for n in model_names]
    best_idx = vals.index(max(vals))
    bar_cols = ['#ffd700' if j == best_idx else colors[j]
                for j in range(len(model_names))]
    bars = ax.bar(model_names, vals, color=bar_cols, edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    gold = mpatches.Patch(color='#ffd700', label='Mejor modelo')
    ax.legend(handles=[gold], fontsize=7)

# Ocultar el subplot extra
axes_flat[5].axis('off')
axes_flat[5].text(0.5, 0.5,
    '⚠️ Con clases desbalanceadas\n(~26% churn):\n\nF1 y ROC-AUC\nson las métricas\nprimarias.\n\nAccuracy puede\nser engañosa.',
    ha='center', va='center', fontsize=11,
    bbox=dict(boxstyle='round', facecolor='#fff3cd', alpha=0.8))

plt.suptitle('Comparación de Métricas — Test Set | Telco Churn',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('metricas_clasificacion.png', bbox_inches='tight')
plt.show()

### 9.2 Curvas ROC — Todos los Modelos

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for (name, y_prob), color in zip(probabilities.items(), colors):
    auc = roc_auc_score(y_test, y_prob)
    RocCurveDisplay.from_predictions(
        y_test, y_prob,
        name=f'{name} (AUC = {auc:.4f})',
        ax=ax, color=color, alpha=0.85
    )

ax.plot([0,1],[0,1],'k--',lw=1.2, label='Random (AUC = 0.5000)')
ax.set_title('Curvas ROC — Comparación de Modelos', fontsize=13, fontweight='bold')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)')
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR / Recall)')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

### 9.3 Matrices de Confusión — Los 4 Modelos

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for (name, y_pred), ax, color in zip(predictions.items(), axes.flatten(), colors):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Churn','Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    f1  = test_results[name]['F1-Score']
    rec = test_results[name]['Recall']
    ax.set_title(f'{name}\nF1={f1:.4f} | Recall={rec:.4f}',
                  fontsize=10, fontweight='bold')

plt.suptitle('Matrices de Confusión — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

### 9.4 Feature Importance — Random Forest

In [ ]:
rf_model  = grid_rf.best_estimator_.named_steps['model']
prep_fit  = grid_rf.best_estimator_.named_steps['prep']
cat_names = prep_fit.named_transformers_['cat']\
                .get_feature_names_out(cat_cols).tolist()
feat_names = num_cols + cat_names + pass_cols

importance = pd.Series(rf_model.feature_importances_, index=feat_names)\
                 .sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bar_colors = ['#ea4335' if v == importance.max() else
               '#ff7043' if v >= importance.quantile(0.75) else
               '#9e9e9e' for v in importance.values]
importance.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white')
ax.axvline(importance.mean(), color='blue', linestyle='--',
            alpha=0.6, label=f'Media: {importance.mean():.4f}')
ax.set_title('Feature Importance — Random Forest Classifier',
              fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia (reducción de impureza Gini)')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance_clf.png', bbox_inches='tight')
plt.show()

print('Top 3 predictores de Churn:')
for f, v in importance.sort_values(ascending=False).head(3).items():
    print(f'  {f}: {v:.4f}')

### 9.5 Reporte de Clasificación Detallado

In [ ]:
best_model_name = max(test_results, key=lambda n: test_results[n]['ROC-AUC'])
print(f'=== REPORTE DETALLADO — {best_model_name} (mejor AUC) ===')
print(classification_report(
    y_test, predictions[best_model_name],
    target_names=['No Churn','Churn'],
    digits=4
))

## 10. Conclusiones

### Comparación de modelos
| Modelo | Fortalezas | Limitaciones |
|---|---|---|
| **Reg. Logística** | Interpretable, calibrado, rápido | Asume linealidad en log-odds |
| **Random Forest** | Robusto, Feature Importance, maneja desbalance | Menos interpretable |
| **Gradient Boosting** | Mayor AUC en tabular data, corrige errores previos | Más lento de entrenar, más hiperparámetros |
| **KNN** | No paramétrico, simple | Lento en inferencia, sensible al desbalance |

### Análisis de métricas
- **ROC-AUC** es la métrica principal: evalúa el modelo en todos los umbrales.
- **Recall** es crítico en churn: detectar clientes que SÍ van a irse es más valioso
  que evitar falsos positivos (el costo de llamar a un cliente que iba a quedarse es bajo).
- **Accuracy** puede ser engañosa con 26% de churn:
  un modelo que siempre predice 'No Churn' tendría ~74% de accuracy pero F1 ≈ 0.
- **Precision** importa si el equipo de retención tiene recursos limitados.

### Desbalance de clases — estrategia aplicada
- `class_weight='balanced'` en Regresión Logística y Random Forest.
- `StratifiedKFold` para preservar la proporción en todos los folds.
- El umbral de decisión (0.5) puede ajustarse según el costo de negocio:
  bajar el umbral aumenta Recall pero reduce Precision.

### Data Leakage — Buenas prácticas
- El `StandardScaler` y `OneHotEncoder` se ajustan solo sobre `X_train`.
- El split es estratificado para representatividad.
- Todo el preprocesamiento está en Pipelines de scikit-learn.

In [ ]:
import joblib
# joblib.dump(grid_rf.best_estimator_, '../app/classification_model.pkl')
print('📦 Modelo listo para exportar con joblib')
print()
best = max(test_results, key=lambda n: test_results[n]['ROC-AUC'])
print(f'=== RESUMEN FINAL — MEJOR MODELO: {best} ===')
for metric, val in test_results[best].items():
    print(f'  {metric:<12}: {val:.4f}')